# Building a FEATHER+ Accelerator with Allo

**Hands-on Tutorial (20 min)**

In this tutorial you will:

1. Learn Allo's **dataflow streaming** primitives (`Stream`, `put`, `get`)
2. Build **spatially-parallel PE arrays** with `meta_for` and `get_pid`
3. Run a complete **FEATHER+ systolic array** simulation
4. Synthesize to **Vitis HLS** and analyze cycle-level performance

**Prerequisites:** Basic Python, familiarity with matrix multiplication.

> **Important:** Make sure you are using the **Python (Allo)** kernel.
> Go to **Kernel → Change Kernel → Python (Allo)** before running any cells.

---
## Section 0: Environment Setup

Run the cell below first to verify that **Allo** and **Vitis HLS** are both available.
If either check fails, make sure you selected the **Python (Allo)** kernel above.

In [ ]:
# ---- Environment Check ----
import subprocess, shutil, os, sys

# 1. Check Allo
try:
    import allo
    from allo.ir.types import int32, Stream
    import allo.dataflow as df
    print(f"[OK] Allo imported successfully")
except ImportError as e:
    print(f"[FAIL] Allo not available: {e}")
    print("  Make sure you are using the 'Python (Allo)' kernel.")

# 2. Check Allo IR (MLIR bindings)
try:
    import allo.ir as air
    print(f"[OK] Allo MLIR bindings available")
except Exception as e:
    print(f"[FAIL] Allo MLIR bindings: {e}")
    print("  LLVM_BUILD_DIR may not be set. Check kernel environment.")

# 3. Check Vitis HLS
vitis_hls = shutil.which("vitis_hls")
if vitis_hls:
    print(f"[OK] Vitis HLS found: {vitis_hls}")
else:
    # Try sourcing Vitis HLS settings
    vitis_settings = "/opt/xilinx/Xilinx_Vivado_Vitis_2022.1/Vitis_HLS/2022.1/settings64.sh"
    if os.path.isfile(vitis_settings):
        result = subprocess.run(
            ["bash", "-c", f"source {vitis_settings} && which vitis_hls"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            vitis_path = result.stdout.strip()
            # Add to PATH for this session
            vitis_bin = os.path.dirname(vitis_path)
            os.environ["PATH"] = vitis_bin + ":" + os.environ.get("PATH", "")
            print(f"[OK] Vitis HLS found (auto-sourced): {vitis_path}")
        else:
            print(f"[WARN] Vitis HLS not on PATH. HLS synthesis (Part 4) will not work.")
            print(f"  Run in a terminal: source {vitis_settings}")
    else:
        print(f"[WARN] Vitis HLS not found. HLS synthesis (Part 4) will not work.")
        print(f"  Simulation exercises (Parts 1-3) will still work fine.")

# 4. Check FEATHER support module
try:
    import numpy as np
    from _support import (
        load_tutorial_trace, print_trace_summary,
        run_feather_simulation, run_feather_csynth,
        Partition,
    )
    print(f"[OK] FEATHER+ support module loaded")
except Exception as e:
    print(f"[FAIL] FEATHER+ support module: {e}")

print("\n--- Environment check complete ---")

---
## Part 1: Allo Dataflow Fundamentals

Allo's dataflow model connects **kernels** via **streams** (hardware FIFOs).
Each kernel runs as an independent hardware process.

| Primitive | What it does |
|---|---|
| `Stream[type, depth]` | Declare a FIFO of given element type and depth |
| `stream.put(value)` | Write a value (blocks if FIFO full) |
| `stream.get()` | Read a value (blocks if FIFO empty) |
| `@df.kernel(mapping=[N])` | Create N parallel hardware copies of a kernel |
| `df.get_pid()` | Get the instance ID (0..N-1) of the current copy |

```
  DRAM                                            DRAM
   |         s_a         +---------+   s_c         |
   +---> [ loader ] ---->| compute |----> [ store ] +
             |    s_b    |  (MAC)  |
             +---------->|         |
                         +---------+
```

### Exercise 1: Stream-Based Dot Product

Build a dataflow kernel that computes the **dot product** of two 4-element vectors.

- `loader` reads A and B from memory, streams values to `compute`
- `compute` reads from streams, multiplies, accumulates — the **MAC unit**
- `store` writes the result to memory

**Your task:** Fill in the `compute` kernel body (replace `pass`).

In [ ]:
_K = 4  # Use underscore prefix to avoid polluting global scope

@df.region()
def dot_product(A: int32[_K], B: int32[_K], C: int32[1]):
    # Declare streams (FIFOs) between kernels
    s_a: Stream[int32, _K]    # A values: loader -> compute
    s_b: Stream[int32, _K]    # B values: loader -> compute
    s_c: Stream[int32, 1]    # Result:   compute -> store

    @df.kernel(mapping=[1], args=[A, B])
    def loader(local_A: int32[_K], local_B: int32[_K]):
        for i in range(_K):
            s_a.put(local_A[i])
            s_b.put(local_B[i])

    @df.kernel(mapping=[1])
    def compute():
        # ============================================
        #  YOUR CODE HERE: Dot product via streams
        #  1. Initialize an accumulator (int32)
        #  2. Loop _K times: read from s_a and s_b,
        #     multiply, accumulate
        #  3. Send the result to s_c via .put()
        # ============================================
        pass  # <-- Replace this

    @df.kernel(mapping=[1], args=[C])
    def store(local_C: int32[1]):
        local_C[0] = s_c.get()

# Build and test
mod = df.build(dot_product, target="simulator")
A_test = np.array([1, 2, 3, 4], dtype=np.int32)
B_test = np.array([4, 3, 2, 1], dtype=np.int32)
C_test = np.zeros(1, dtype=np.int32)
mod(A_test, B_test, C_test)

print(f"A = {A_test}")
print(f"B = {B_test}")
print(f"A . B = {C_test[0]}  (expected: {np.dot(A_test, B_test)})")
assert C_test[0] == np.dot(A_test, B_test), "FAIL"
print("PASS")
del _K  # Clean up

<details>
<summary><b>Click to reveal answer</b></summary>

Replace `pass` in the `compute` kernel with:

```python
        accum: int32 = 0
        for i in range(_K):
            a: int32 = s_a.get()
            b: int32 = s_b.get()
            accum += a * b
        s_c.put(accum)
```

**Note:** The `loader` and `store` kernels use `args=[A, B]` / `args=[C]` to declare
which DRAM arrays they access. In Allo dataflow, every kernel that reads/writes DRAM
must list those arrays in `args=` and declare them as typed local parameters.

</details>

---
## Part 2: Spatial Parallelism

Real accelerators use **many PEs in parallel**. Allo makes this easy:

| Primitive | Effect |
|---|---|
| `@df.kernel(mapping=[N])` | N **independent hardware copies** of the kernel |
| `df.get_pid()` | Returns each copy's unique ID (0 to N-1) |
| `allo.meta_for(N) as i` | Unrolls a loop into N **parallel** hardware operations |
| `Stream[type, depth][N]` | Array of N streams — one per PE |

```
                +------+   s_c[0]
  s_a[0] ------>| PE 0 |---------->
  s_b[0] ------>|      |
                +------+
                +------+   s_c[1]
  s_a[1] ------>| PE 1 |---------->
  s_b[1] ------>|      |
                +------+
                +------+   s_c[2]
  s_a[2] ------>| PE 2 |---------->
  s_b[2] ------>|      |
                +------+
                +------+   s_c[3]
  s_a[3] ------>| PE 3 |---------->
  s_b[3] ------>|      |
                +------+
```

### Exercise 2: Parallel PE Array

Scale up to **4 parallel PEs**, each computing one element of a vector-matrix product:

$$C[4] = A[4] \times B[4 \times 4]$$

Each PE computes: `C[col] = sum_k A[k] * B[k, col]`

**Your task:** Fill in the `compute` kernel (replace `pass`).

Hints:
- `pid = df.get_pid()` tells you which PE you are (0-3)
- Read from `s_a[pid]` and `s_b[pid]`
- Output to `s_c[pid]`

In [ ]:
_K, _N = 4, 4  # Use underscore prefix to avoid polluting global scope

@df.region()
def parallel_mac(A: int32[_K], B: int32[_K, _N], C: int32[_N]):
    s_a: Stream[int32, _K][_N]     # A broadcast to each PE
    s_b: Stream[int32, _K][_N]     # B column per PE
    s_c: Stream[int32, 1][_N]     # Output per PE

    @df.kernel(mapping=[1], args=[A, B])
    def loader(local_A: int32[_K], local_B: int32[_K, _N]):
        for k in range(_K):
            with allo.meta_for(_N) as col:  # 4 parallel writes
                s_a[col].put(local_A[k])         # Same A value to all PEs
                s_b[col].put(local_B[k, col])    # Different B column per PE

    @df.kernel(mapping=[_N])        # <-- 4 parallel PEs!
    def compute():
        pid = df.get_pid()
        # ============================================
        #  YOUR CODE HERE: MAC unit (5 lines)
        #  Same as Exercise 1, but use
        #    s_a[pid].get() and s_b[pid].get()
        #  Output to s_c[pid].put(...)
        # ============================================
        pass  # <-- Replace this

    @df.kernel(mapping=[1], args=[C])
    def store(local_C: int32[_N]):
        with allo.meta_for(_N) as col:
            local_C[col] = s_c[col].get()

# Build and test
mod = df.build(parallel_mac, target="simulator")
A_test = np.array([1, 2, 3, 4], dtype=np.int32)
B_test = np.eye(4, dtype=np.int32) * 3
C_test = np.zeros(4, dtype=np.int32)
mod(A_test, B_test, C_test)

expected = A_test @ B_test
print(f"A     = {A_test}")
print(f"B     = diagonal({B_test[0,0]})")
print(f"A x B = {C_test}  (expected: {expected})")
assert np.array_equal(C_test, expected), "FAIL"
print("PASS")
del _K, _N  # Clean up

<details>
<summary><b>Click to reveal answer</b></summary>

Replace `pass` in the `compute` kernel with:

```python
        accum: int32 = 0
        for k in range(_K):
            a: int32 = s_a[pid].get()
            b: int32 = s_b[pid].get()
            accum += a * b
        s_c[pid].put(accum)
```

</details>

---
## Part 2.5: From Independent PEs to Connected Arrays (SPMW)

In Exercises 1-2, each PE worked **independently** — they all read from separate streams
and never communicated with each other. But real accelerators have PEs that are
**tightly interconnected**: data flows *through* the array, from one PE to the next.

This is the key idea behind **SPMW (Single Program, Multiple Connected Work Units)**:

| SPMD (e.g., CUDA) | SPMW (Allo) |
|---|---|
| Independent threads | **Connected** work units |
| Shared memory for communication | **Streams** (FIFOs) between neighbors |
| All threads run the same code | One program, **specialized** per position via `meta_if` |

In SPMW, you write **one program** that describes all PEs over a grid.
Each PE uses `df.get_pid()` to know its position, and `meta_if` to specialize its behavior
(e.g., edge PEs load data, interior PEs forward data to neighbors).

### The Systolic Array Pattern

The classic example is a **systolic array** for GEMM. Data flows through the array:
- **A values** stream left-to-right across each row
- **B values** stream top-to-bottom down each column
- Each PE reads A and B from its neighbor, does a MAC, and **forwards** both values onward

```
      B[0]    B[1]
       |        |
       v        v
A[0]->[ PE ]->[ PE ]-> (drain)
       |        |
       v        v
A[1]->[ PE ]->[ PE ]-> (drain)
       |        |
       v        v
     (drain)  (drain)
```

In Allo SPMW, this entire 2D array is described in ~36 lines:

```python
@df.kernel(grid=[P0, P1])
def gemm(A: int8[M, K], B: int8[K, N], C: int16[M, N]):
    i, j = df.get_pid()

    # Edge loaders: feed data into the array
    with allo.meta_if(j == 0):        # Left column: stream A rows
        for k in range(K):
            fifo_A[i, j+1].put(A[i, k])

    with allo.meta_elif(i == 0):      # Top row: stream B columns
        for k in range(K):
            fifo_B[i+1, j].put(B[k, j])

    # Interior PEs: read, compute, forward
    with allo.meta_else():
        c: int16 = 0
        for k in range(K):
            a = fifo_A[i, j].get()     # From left neighbor
            b = fifo_B[i, j].get()     # From top neighbor
            c += a * b                 # MAC
            fifo_A[i, j+1].put(a)      # Forward A rightward
            fifo_B[i+1, j].put(b)      # Forward B downward
        C[i-1, j-1] = c
```

**Key SPMW concepts:**
1. **One program** describes all work units over a grid (`grid=[P0, P1]`)
2. **Type-safe streams** specify connectivity among work units (`fifo_A[i, j].get()` / `.put()`)
3. **`meta_if`** specializes behavior per grid position (loaders vs compute PEs)

This is exactly how FEATHER+ works in Part 3 — the PE array uses `mapping=[AH+1, AW]`
with `meta_if` to specialize rows, and streams to forward data down columns.
The result: **220 lines of Allo** replace **2941 lines of Verilog RTL**, with comparable performance.

---
## Part 3: FEATHER+ Full Accelerator

Now let's see these concepts at work in a **real accelerator**.

FEATHER+ is a reconfigurable DNN accelerator with flexible dataflow support.
The high-level architecture has four main components:

<img src="figs/FEATHER.png" width="300">

- **Stationary Buffer / Streaming Buffer**: Feed weights and activations into the array
- **NEST**: The PE array — reconfigurable compute with flexible dataflow (output/weight/input stationary)
- **BIRRD**: Butterfly network for reduction and reordering of partial results
- **Output Buffer**: Collects results, feeds them as inputs to the next layer

Our Allo implementation maps this to **7 pipelined dataflow kernels** on a **4x4 PE array** (16 PEs):

```
  DRAM                                                 DRAM
   |                                                    ^
   v                                                    |
 +----------+  col_a_in   +---------------------+   +---------------+
 | a_loader |------------>|                     |   |               |
 +----------+             |   pe_array [5 x 4]  |   |               |
                          |                     |   |               |
 +----------+  col_w_in   |  +--+--+--+--+     |   |               |
 | w_loader |------+      |  |PE|PE|PE|PE| row0 |   |               |
 +----------+      |      |  +--+--+--+--+     |   |               |
              +-----v---+ |  |PE|PE|PE|PE| row1 |   | output_accum  |
              |w_broad- | |  +--+--+--+--+     |   |               |
              |cast [4] |--->|PE|PE|PE|PE| row2 |   |               |
              +---------+ |  +--+--+--+--+     |   |               |
                          |  |PE|PE|PE|PE| row3 |   |               |
                          |  +--+--+--+--+     |   |               |
                          |  |G |G |G |G | gath.|   |               |
                          |  +--+--+--+--+     |   |               |
                          +--------+-----------+   |               |
                                   |               |               |
                          +--------v-----------+   |               |
   +---------+  inst      |  BIRRD [3 x 2]    |   |               |
   | inst_rw |----------->| (butterfly reduce) |-->|               |
   +---------+            +--------------------+   +-------+-------+
                                                           |
                                                           v
                                                        C [M, N]
```

**Key patterns you already learned:**
- Each PE uses `stream.get()` / `stream.put()` (Exercise 1)
- The PE array uses `mapping=[5, 4]` for 20 parallel instances (Exercise 2)
- `meta_if` / `meta_else` specializes behavior per PE row

### FEATHER+ PE Array — Key Code

The NEST is the compute core of FEATHER+. It consists of an AH x AW grid of PEs,
each with local registers for weights and a MAC unit. Activations stream down the
columns, while weights are broadcast from the crossbar:

<img src="figs/NEST.png" width="400">

After the PE array computes partial products, results flow into the **BIRRD**
(Butterfly Interconnect for Reduction and ReorDering) network. BIRRD uses
configurable butterfly switches to reduce and rearrange outputs:

<img src="figs/BIRRD.png" width="400">

Here is the PE array kernel in Allo, using the same patterns from the exercises:

```python
@df.kernel(mapping=[AH + 1, AW])     # 5x4 = 20 instances
def pe_array():
    ni, nj = df.get_pid()             # (row, col) of this PE

    with allo.meta_if(ni == AH):      # Row 4: Gather unit
        # Collect results from compute PEs, send to BIRRD
        ...

    with allo.meta_else():            # Rows 0-3: Compute PEs
        for _op in range(total_ops):
            tile_accum: int32 = 0
            for nk in range(AH):
                # Column-streaming: row 0 reads from DRAM loader,
                # rows 1+ read from the PE above
                a_val: int32 = 0
                with allo.meta_if(ni == 0):
                    a_val = col_a_in[nj].get()           # From a_loader
                with allo.meta_else():
                    a_val = pe_a_down[ni - 1, nj].get()  # From PE above

                w_val: int32 = pe_w_in[ni, nj].get()     # From w_broadcast

                # Forward A to next row (column-streaming)
                with allo.meta_if(ni < AH - 1):
                    pe_a_down[ni, nj].put(a_val)

                tile_accum += a_val * w_val  # <-- MAC!

            pe_out[ni, nj].put(tile_accum)
```

Notice how `meta_if(ni == 0)` generates **different hardware** for row 0 vs other rows,
while `tile_accum += a_val * w_val` is the same MAC you wrote in Exercise 1.

### Run FEATHER+ Simulation

Let's run the full 4x4 FEATHER+ on a real GEMM workload from the paper:

**Figure 7:** `C[16, 8] = A[16, 12] x B[12, 8]`

In [ ]:
# Load the workload trace
trace = load_tutorial_trace()
print_trace_summary(trace)

In [ ]:
# Build and run the full FEATHER+ simulation (~10s)
C_result, passed = run_feather_simulation(trace)

---
## Part 4: HLS Synthesis

To turn our Allo design into **real hardware**, we synthesize with Vitis HLS.

The key optimization is **scheduling** — telling HLS how to pipeline loops
and partition arrays for parallel access:

| Directive | Effect |
|---|---|
| `s.pipeline("kernel:loop")` | Pipeline a loop for maximum throughput |
| `s.partition("kernel:array", dim=N, ...)` | Split an array into parallel banks |
| `Partition.Complete` | Every element gets its own port (registers) |
| `Partition.Cyclic` | Round-robin distribution across banks |

### Exercise 3: Write the HLS Schedule

Fill in the scheduling directives to optimize the FEATHER+ kernel.

**What to optimize:**
1. **Pipeline** the `w_loader`'s tile loop so tiles overlap
2. **Partition** the output C along columns for parallel writes
3. **Partition** input A along rows for AW parallel reads per cycle
4. **Partition** weight B completely (small enough for registers)
5. **Partition** the accumulator buffer for parallel access

In [ ]:
def my_schedule(s, K, N, AH, AW):
    """HLS scheduling directives for FEATHER+."""
    # ============================================
    #  YOUR CODE HERE
    # ============================================

    # 1. Pipeline the w_loader tile loop
    #    s.pipeline("w_loader_0:tile")

    # 2. Partition C completely along columns for parallel output writes
    #    s.partition("full_matrix_top:C", dim=2,
    #                partition_type=Partition.Complete)

    # 3. Partition A along rows (dim=1) for AW parallel reads
    #    s.partition("a_loader_0:local_A", dim=1, factor=AW,
    #                partition_type=Partition.Cyclic)

    # 4. Partition A along columns (dim=2) completely
    #    s.partition("a_loader_0:local_A", dim=2,
    #                partition_type=Partition.Complete)

    # 5. Partition B completely (both dims) for register storage
    #    s.partition("w_loader_0:local_B", dim=1,
    #                partition_type=Partition.Complete)
    #    s.partition("w_loader_0:local_B", dim=2,
    #                partition_type=Partition.Complete)

    # 6. Partition accumulator buffer for parallel access
    #    s.partition("output_accum_0:tile_acc", dim=1,
    #                partition_type=Partition.Complete)
    #    s.partition("output_accum_0:tile_acc", dim=2,
    #                partition_type=Partition.Complete)

    pass  # <-- Remove this after uncommenting the directives above

<details>
<summary><b>Click to reveal answer</b></summary>

Uncomment all the directives and remove `pass`:

```python
def my_schedule(s, K, N, AH, AW):
    s.pipeline("w_loader_0:tile")
    s.partition("full_matrix_top:C", dim=2,
                partition_type=Partition.Complete)
    s.partition("a_loader_0:local_A", dim=1, factor=AW,
                partition_type=Partition.Cyclic)
    s.partition("a_loader_0:local_A", dim=2,
                partition_type=Partition.Complete)
    s.partition("w_loader_0:local_B", dim=1,
                partition_type=Partition.Complete)
    s.partition("w_loader_0:local_B", dim=2,
                partition_type=Partition.Complete)
    s.partition("output_accum_0:tile_acc", dim=1,
                partition_type=Partition.Complete)
    s.partition("output_accum_0:tile_acc", dim=2,
                partition_type=Partition.Complete)
```

</details>

In [ ]:
# Run HLS synthesis (requires Vitis HLS, ~2 minutes)
run_feather_csynth(trace, my_schedule)

---
## Summary

In this tutorial you built a FEATHER+ accelerator using Allo:

| Concept | Allo Primitive | Where |
|---|---|---|
| Dataflow streaming | `Stream`, `.put()`, `.get()` | Exercise 1 |
| Spatial parallelism | `mapping=[N]`, `meta_for(N)` | Exercise 2 |
| Compile-time specialization | `meta_if` / `meta_else` | PE array walkthrough |
| HLS scheduling | `pipeline`, `partition` | Exercise 3 |

The full 4x4 FEATHER+ achieves **~537 cycles** for a 16x12x8 GEMM at **411 MHz**.
The same architecture scales to **16x16 arrays** for production workloads.

**Next steps:**
- Try different `Partition` strategies and see how they affect cycle count
- Look at `feather_minisa.py` in `/opt/feather_tutorial/allo-feather/` for the full kernel source
- Run RTL co-simulation with `--hls cosim` for cycle-accurate verification